# V6 — 07: Eval Main Models (A3, P1)

| Tag | Policy           | m=50 | m=100 |
|-----|------------------|------|-------|
| A3  | ACM3+ICPE        | ✓    | ✓     |
| P1  | ACM3+ICPE+SSCP   | ✓    | ✓     |

2 models × 2 GPUs.  m=50 and m=100 launched in separate rounds.

In [ ]:
import sys, os
from pathlib import Path

GPU_OFFSET = 0  # ← only line to change

sys.path.insert(0, str(Path(".").resolve()))
import common_v6 as v6

os.environ.setdefault("MUJOCO_GL", "osmesa")
os.environ.setdefault("PYOPENGL_PLATFORM", "osmesa")

TAGS = ["A3", "P1"]
EVAL_JOBS_R1 = [(tag, i, 50)  for i, tag in enumerate(TAGS)]
EVAL_JOBS_R2 = [(tag, i, 100) for i, tag in enumerate(TAGS)]

In [ ]:
print("=== Training status ===")
v6.print_training_status(TAGS)
print()
print("=== Eval status ===")
v6.print_eval_status(TAGS)

In [ ]:
print("Launching Round 1 (m=50) ...")
procs_r1 = v6.launch_eval(EVAL_JOBS_R1, GPU_OFFSET)
for tag in TAGS:
    print(f"  tail -f {v6.get_output_dir(tag) / 'eval' / 'eval.log'}")

In [ ]:
print("Launching Round 2 (m=100) ...")
procs_r2 = v6.launch_eval(EVAL_JOBS_R2, GPU_OFFSET)
for tag in TAGS:
    print(f"  tail -f {v6.get_output_dir(tag) / 'eval' / 'eval.log'}")

In [ ]:
results = {}
for tag in TAGS:
    m = v6.build_metrics_from_eval_info(tag)
    results[tag] = m
    v6.save_metrics(tag, m)

print(f"{'TAG':<8} {'LABEL':<35} {'SR':>6} {'CI_LO':>7} {'CI_HI':>7}")
print("-" * 70)
for tag in TAGS:
    m = results.get(tag, {})
    sr_s = f"{m['sr']:.3f}"     if m.get('sr')       is not None else " ─"
    lo_s = f"{m['sr_ci_lo']:.3f}" if m.get('sr_ci_lo') is not None else " ─"
    hi_s = f"{m['sr_ci_hi']:.3f}" if m.get('sr_ci_hi') is not None else " ─"
    print(f"{tag:<8} {m.get('label', tag):<35} {sr_s:>6} {lo_s:>7} {hi_s:>7}")